# PhageMatch-PK - Kleborate typing (self-contained)

Types 500 clinical *K. pneumoniae* genomes - International comparison K. pneumoniae:
ST, **K-locus (capsule)**, O-locus, AMR, virulence.

Accessions are embedded below - no uploads needed. Run cells in order.
**Cell 3 restarts the runtime on purpose**; resume at Cell 4 afterwards.


## Cell 1 - System tools

In [ ]:
%%bash
# No `set -e`: keep diagnostics visible even if one probe fails.
# mash is REQUIRED - Kleborate's species check shells out to the mash
# binary and aborts with 'could not find mash'. The pip package named
# 'mash' is a different project and does NOT provide the executable.
apt-get -qq update > /dev/null 2>&1
apt-get -qq install -y minimap2 ncbi-blast+ mash > /dev/null 2>&1
curl -sSL -o /usr/local/bin/datasets \
  https://ftp.ncbi.nlm.nih.gov/pub/datasets/command-line/v2/linux-amd64/datasets
chmod +x /usr/local/bin/datasets
echo '--- versions ---'
minimap2 --version || echo 'minimap2 MISSING'
datasets --version || echo 'datasets MISSING'
mash --version || echo 'mash MISSING'


## Cell 2 - Install Kleborate + Kaptive

**Kaptive is pinned to 3.2.2 on purpose.** Kleborate 3.2.4 declares an unpinned
`kaptive` dependency, but Kaptive 3.3.0 restructured its package and removed
`kaptive.database`, which Kleborate's capsule module imports. Installing latest
Kaptive gives `ModuleNotFoundError: No module named 'kaptive.database'`.
3.2.2 is the newest release that still provides it.


In [ ]:
import subprocess, sys

KAPTIVE_PIN = 'kaptive==3.2.2'   # see note above - do not unpin

def npver():
    r = subprocess.run([sys.executable, '-c', 'import numpy; print(numpy.__version__)'],
                       capture_output=True, text=True)
    return r.stdout.strip() or '(none)'

before = npver()
print('numpy before:', before)
r = subprocess.run([sys.executable, '-m', 'pip', 'install', 'kleborate', KAPTIVE_PIN],
                   capture_output=True, text=True)
print(r.stdout[-3000:])
if r.returncode != 0:
    print('PIP FAILED:', r.stderr[-3000:])
print('numpy after :', npver())


## Cell 3 - Restart runtime (expected; resume at Cell 4)

In [ ]:
import os
print('Restarting - this is expected. Continue at Cell 4.')
os.kill(os.getpid(), 9)


## Cell 4 - Verify install, detect CLI

In [ ]:
import subprocess

for mod in ['numpy', 'numba', 'kaptive', 'kleborate']:
    try:
        m = __import__(mod)
        print(f'{mod:<10}', getattr(m, '__version__', 'ok'))
    except Exception as e:
        print(f'{mod:<10} FAILED: {type(e).__name__}: {e}')

h = subprocess.run(['kleborate', '--help'], capture_output=True, text=True)
help_text = h.stdout + h.stderr
print(help_text[:4000])

for flag in ['--list-presets', '--list-modules']:
    r = subprocess.run(['kleborate', flag], capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    if r.returncode == 0 and out:
        print(f'\n=== {flag} ===\n' + out[:2500])


## Cell 4b - Mount Google Drive (crash protection)

Colab free runtimes get reclaimed without warning, and anything in `/content`
dies with them. Typing takes ~50 min, so we keep the **results and per-chunk
progress markers on Drive** - a few MB. The 1.5 GB of genome FASTAs stay in
local scratch because re-downloading them costs only ~45 s.

Net effect: if the session dies, rerun Cells 1-7 and the typing resumes from
the last finished chunk instead of starting over.

**This cell asks for Google Drive access - approve it in the popup.**


In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

PERSIST = Path('/content/drive/MyDrive/phagematch-pk-comparison')
PERSIST.mkdir(parents=True, exist_ok=True)
print('persisting results to:', PERSIST)

existing = sorted(PERSIST.rglob('chunk_*.done'))
print(f'chunks already finished from a previous session: {len(existing)}')


## Cell 5 - Download 500 genomes from NCBI

In [ ]:
import shutil, subprocess, zipfile
from pathlib import Path

accessions = ["GCA_040293825.1","GCA_050519405.1","GCA_052974865.2","GCA_022324405.1","GCA_036468375.1","GCA_033136245.1","GCA_021976945.1","GCA_049687355.1","GCA_048052295.1","GCA_047922515.1","GCA_059667425.1","GCA_034740195.1","GCA_021899215.1","GCA_050518645.1","GCA_055001825.1","GCA_027608655.1","GCA_023544765.1","GCA_022143925.4","GCA_031013485.2","GCA_027827455.1","GCA_040294445.1","GCA_027625495.1","GCA_052043565.1","GCA_009722505.1","GCA_051901695.1","GCA_047967325.1","GCA_047965775.1","GCA_015626945.1","GCA_017974115.1","GCA_956476105.1","GCA_055001805.1","GCA_029577035.1","GCA_057412115.1","GCA_053067065.1","GCA_021963105.1","GCA_054198665.1","GCA_046298035.1","GCA_023554775.1","GCA_032744215.1","GCA_050414095.1","GCA_025378805.4","GCA_037082055.1","GCA_016918985.1","GCA_059605455.1","GCA_047967745.1","GCA_032334695.1","GCA_026623615.1","GCA_050517585.1","GCA_050414475.1","GCA_034710635.1","GCA_040293745.1","GCA_050415355.1","GCA_056656495.1","GCA_032044895.1","GCA_059364755.1","GCA_050413955.1","GCA_052184675.1","GCA_033119485.1","GCA_059089665.1","GCA_051104655.1","GCA_055001665.1","GCA_049204205.1","GCA_053198545.1","GCA_040291985.1","GCA_040293025.1","GCA_011742225.1","GCA_032135695.1","GCA_050090855.1","GCA_022324585.1","GCA_016762355.1","GCA_015643805.1","GCA_037862715.1","GCA_057411225.1","GCA_034487265.1","GCA_051102015.1","GCA_043478175.1","GCA_057947615.1","GCA_023208695.2","GCA_047225805.1","GCA_021947595.1","GCA_042993835.1","GCA_040291115.1","GCA_905329775.2","GCA_040291075.1","GCA_022354945.1","GCA_047922635.1","GCA_059878175.1","GCA_050245495.1","GCA_048973125.1","GCA_053513345.1","GCA_040293565.1","GCA_027616745.1","GCA_040436295.1","GCA_024491595.1","GCA_905232385.2","GCA_028913845.4","GCA_039613065.1","GCA_022168265.1","GCA_040293305.1","GCA_018438945.1","GCA_022053205.1","GCA_048955995.1","GCA_016762515.1","GCA_034397115.1","GCA_047921945.1","GCA_027617205.1","GCA_011742285.2","GCA_052868015.1","GCA_047766555.1","GCA_022150845.1","GCA_031014865.2","GCA_034432035.1","GCA_037082615.1","GCA_047964925.1","GCA_022432765.1","GCA_011742045.1","GCA_050868855.1","GCA_024378855.1","GCA_025779635.1","GCA_022028735.1","GCA_902704205.1","GCA_045566935.1","GCA_022323985.1","GCA_025874965.1","GCA_045753215.1","GCA_052848215.1","GCA_027625755.1","GCA_059131855.1","GCA_034085685.1","GCA_048931655.1","GCA_057947235.1","GCA_058911805.1","GCA_043476535.1","GCA_051300775.1","GCA_025378905.4","GCA_059670405.1","GCA_043477735.1","GCA_059653695.1","GCA_051105575.1","GCA_028450985.4","GCA_022262635.1","GCA_052446395.1","GCA_045486735.1","GCA_052187435.1","GCA_044444125.1","GCA_037543195.1","GCA_056112145.1","GCA_053648675.1","GCA_043830535.1","GCA_040287765.1","GCA_050016475.1","GCA_056136545.1","GCA_016753415.1","GCA_050517225.1","GCA_023893365.1","GCA_055005285.1","GCA_024586725.1","GCA_903993085.2","GCA_055002545.1","GCA_038376695.1","GCA_027893055.1","GCA_017975445.1","GCA_047967085.1","GCA_054967645.1","GCA_027611725.1","GCA_059026745.1","GCA_057430215.1","GCA_043475715.1","GCA_052867415.1","GCA_052135475.1","GCA_014834805.1","GCA_056710275.1","GCA_040291305.1","GCA_053817465.1","GCA_024238855.1","GCA_021946415.1","GCA_036469295.1","GCA_031018285.2","GCA_042918665.1","GCA_037082605.1","GCA_040688545.1","GCA_050016415.1","GCA_028169265.1","GCA_007996225.1","GCA_029953105.1","GCA_022325985.1","GCA_055414765.1","GCA_027610505.1","GCA_022308835.1","GCA_016762625.1","GCA_056133785.1","GCA_034395215.1","GCA_027893075.1","GCA_022110715.1","GCA_027616825.1","GCA_047922155.1","GCA_025377115.1","GCA_900173405.1","GCA_032637385.1","GCA_040293185.1","GCA_041346185.1","GCA_040066815.1","GCA_027616225.1","GCA_032236475.2","GCA_040290525.1","GCA_056870805.1","GCA_040294085.1","GCA_020829925.1","GCA_021962495.1","GCA_032072625.1","GCA_049207605.1","GCA_022316425.1","GCA_054584435.1","GCA_047760655.1","GCA_022323925.1","GCA_053513865.1","GCA_048977725.1","GCA_056503875.1","GCA_043475745.1","GCA_047275905.1","GCA_050517525.1","GCA_045753355.1","GCA_050510205.1","GCA_947573095.1","GCA_036471655.1","GCA_040715595.1","GCA_034398435.1","GCA_056657755.1","GCA_056133205.1","GCA_059088775.1","GCA_037956705.1","GCA_002852695.1","GCA_016762555.1","GCA_050516965.1","GCA_027612085.1","GCA_050415775.1","GCA_056637075.1","GCA_045488575.1","GCA_022262775.1","GCA_056656235.1","GCA_027617125.1","GCA_022324185.1","GCA_031624735.1","GCA_056544625.1","GCA_022156945.1","GCA_051686985.1","GCA_956476505.1","GCA_057480775.1","GCA_902704095.1","GCA_055000885.1","GCA_037392825.2","GCA_052206675.1","GCA_047919975.1","GCA_042443095.2","GCA_040560465.1","GCA_059209895.1","GCA_055671625.1","GCA_024197795.1","GCA_019401055.1","GCA_022162145.1","GCA_025875585.1","GCA_058827965.1","GCA_902508925.1","GCA_902649915.1","GCA_022157425.1","GCA_027611945.1","GCA_905329825.2","GCA_036248965.1","GCA_056639755.1","GCA_047919435.1","GCA_023315495.2","GCA_008374195.1","GCA_058065675.1","GCA_050507705.1","GCA_047922315.1","GCA_052986515.1","GCA_057948115.1","GCA_023660375.1","GCA_046036615.1","GCA_021953325.1","GCA_053068545.1","GCA_049551475.1","GCA_050581025.1","GCA_047968905.1","GCA_040078235.1","GCA_019837365.1","GCA_027893215.1","GCA_055781205.1","GCA_003227555.1","GCA_902704515.1","GCA_050416435.1","GCA_034137805.1","GCA_052867355.1","GCA_031041695.3","GCA_024492975.1","GCA_053514085.1","GCA_055420425.1","GCA_022328325.1","GCA_031038135.3","GCA_022198625.1","GCA_057662305.1","GCA_019220375.1","GCA_040306705.1","GCA_047969255.1","GCA_040120905.1","GCA_056740495.1","GCA_060046345.1","GCA_022325825.1","GCA_045666085.1","GCA_039545765.1","GCA_055416265.1","GCA_019317005.1","GCA_057032315.1","GCA_051098175.1","GCA_045482395.1","GCA_058825705.1","GCA_037862515.1","GCA_054167295.1","GCA_016762565.1","GCA_051663765.1","GCA_059408345.1","GCA_050415815.1","GCA_047966105.1","GCA_059186525.1","GCA_026251835.4","GCA_051843595.1","GCA_047964425.1","GCA_019249455.1","GCA_022109895.1","GCA_037862675.1","GCA_045278765.1","GCA_047967165.1","GCA_021958945.1","GCA_040293785.1","GCA_052043605.1","GCA_033243425.1","GCA_040293885.1","GCA_022116415.1","GCA_053513765.1","GCA_021916885.1","GCA_027611845.1","GCA_040292285.1","GCA_027614985.1","GCA_049532545.1","GCA_040294405.1","GCA_047967985.1","GCA_027893475.1","GCA_051301955.1","GCA_057593695.1","GCA_045720535.1","GCA_050167065.1","GCA_057412055.1","GCA_047998705.1","GCA_053197445.1","GCA_027617105.1","GCA_048977765.1","GCA_051776295.1","GCA_050414355.1","GCA_034720215.1","GCA_057428845.1","GCA_947576785.1","GCA_034734055.1","GCA_043478255.1","GCA_027865765.1","GCA_027616885.1","GCA_047919015.1","GCA_055003005.1","GCA_947576335.1","GCA_025223495.4","GCA_040292485.1","GCA_021928075.1","GCA_032044435.1","GCA_023505665.1","GCA_030845815.1","GCA_057032115.1","GCA_023275295.1","GCA_031045735.2","GCA_050016455.1","GCA_040293405.1","GCA_024237475.1","GCA_048933015.1","GCA_056637215.1","GCA_034737055.1","GCA_044901765.1","GCA_021727875.1","GCA_057438495.1","GCA_057429535.1","GCA_050428555.1","GCA_055977295.1","GCA_053079325.1","GCA_027617745.1","GCA_048968465.1","GCA_050016195.1","GCA_024491455.1","GCA_053060625.1","GCA_022554615.1","GCA_050455325.1","GCA_043569555.1","GCA_030216205.1","GCA_051100465.1","GCA_040293165.1","GCA_019149405.1","GCA_034406825.1","GCA_040715815.1","GCA_034508835.1","GCA_045729485.1","GCA_047965355.1","GCA_051089105.1","GCA_040291205.1","GCA_047306425.1","GCA_059089025.1","GCA_048000105.1","GCA_059675545.1","GCA_021976235.1","GCA_022354925.1","GCA_025754775.1","GCA_058681535.1","GCA_059663465.1","GCA_054999775.1","GCA_053709135.1","GCA_021795195.1","GCA_026346375.1","GCA_059295015.1","GCA_050413915.1","GCA_037391015.1","GCA_048816085.1","GCA_046578995.1","GCA_052868875.1","GCA_058315155.1","GCA_037082665.1","GCA_052196855.1","GCA_947573215.1","GCA_053813265.1","GCA_053063725.1","GCA_051113915.1","GCA_033117335.1","GCA_022353345.1","GCA_956476515.1","GCA_059674765.1","GCA_956481075.1","GCA_034744295.1","GCA_056639995.1","GCA_032066425.1","GCA_054522415.1","GCA_051330125.1","GCA_050518185.1","GCA_040561215.1","GCA_052598455.1","GCA_050414135.1","GCA_011742185.1","GCA_043478395.1","GCA_050507405.1","GCA_036527975.1","GCA_055413965.1","GCA_032153355.1","GCA_027893195.1","GCA_040306825.1","GCA_021958005.1","GCA_050607665.1","GCA_041926565.1","GCA_040294325.1","GCA_011742825.1","GCA_059026955.1","GCA_046039055.1","GCA_047918855.1","GCA_045566715.1","GCA_022149865.1","GCA_041767205.1","GCA_024359405.1","GCA_054997705.1","GCA_028219995.1","GCA_051105335.1","GCA_047922375.1","GCA_058343855.1","GCA_021962955.1","GCA_030675775.1","GCA_035781115.1","GCA_042973115.1","GCA_044009815.1","GCA_050624505.1","GCA_047920435.1","GCA_055975925.1","GCA_053904885.1","GCA_033193695.1","GCA_050416235.1","GCA_905217795.2","GCA_021809145.1","GCA_051329945.1","GCA_022141685.1","GCA_046625385.1","GCA_022324285.1","GCA_053225765.1","GCA_027610725.1","GCA_021962135.1","GCA_051298935.1","GCA_021520395.1","GCA_055416365.1","GCA_905219335.2","GCA_034523495.1","GCA_022325335.1"]
print(len(accessions), 'accessions')

WORK = Path('/content/work'); ASM = WORK / 'assemblies'
ASM.mkdir(parents=True, exist_ok=True)
BATCH = 25
batches = [accessions[i:i+BATCH] for i in range(0, len(accessions), BATCH)]

for n, batch in enumerate(batches, 1):
    marker = WORK / f'.b{n}.done'
    if marker.exists():
        print(f'batch {n}/{len(batches)} cached'); continue
    lf = WORK / f'b{n}.txt'; lf.write_text('\n'.join(batch) + '\n')
    zp = WORK / f'b{n}.zip'
    res = subprocess.run(['datasets','download','genome','accession',
                          '--inputfile',str(lf),'--include','genome',
                          '--filename',str(zp),'--no-progressbar'],
                         capture_output=True, text=True)
    if res.returncode != 0 or not zp.exists():
        print(f'batch {n} FAILED:', res.stderr.strip()[:250]); continue
    with zipfile.ZipFile(zp) as zf:
        for m in zf.namelist():
            if not m.endswith(('.fna','.fa','.fasta')):
                continue
            parts = m.split('/')
            acc = parts[2] if len(parts) >= 3 else Path(m).stem
            with zf.open(m) as s, (ASM / f'{acc}.fna').open('wb') as d:
                shutil.copyfileobj(s, d)
    zp.unlink(); marker.touch()
    print(f'batch {n}/{len(batches)} ok ({len(list(ASM.glob("*.fna")))} genomes)')

fastas = sorted(ASM.glob('*.fna'))
print(f'\nDownloaded {len(fastas)}/{len(accessions)} '
      f'({sum(f.stat().st_size for f in fastas)/1e6:.0f} MB)')


## Cell 6 - Smoke test (finds the working CLI form)

In [ ]:
import subprocess
from pathlib import Path

TEST = WORK / 'smoke'; TEST.mkdir(exist_ok=True)
one = sorted(ASM.glob('*.fna'))[0]
print('test genome:', one.name)

variants = [
    ('v3 preset kpsc', ['kleborate','-a',str(one),'-o',str(TEST/'t1'),'-p','kpsc']),
    ('v3 preset kp',   ['kleborate','-a',str(one),'-o',str(TEST/'t2'),'-p','kp']),
    ('v3 module kpsc', ['kleborate','-a',str(one),'-o',str(TEST/'t3'),
                        '-m','klebsiella_pneumo_complex']),
    ('v2 --all',       ['kleborate','-a',str(one),'--all','-o',str(TEST/'t4.txt')]),
]

WORKING_CMD = None
for label, cmd in variants:
    print('--- trying:', label)
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode == 0:
        print('    OK'); WORKING_CMD = label; break
    print('    rc=', r.returncode)
    print('    stderr:', (r.stderr or '').strip()[-700:] or '(none)')

if WORKING_CMD is None:
    raise SystemExit('All variants failed - send the output above.')
print('\n>>> using:', WORKING_CMD)

import pandas as pd
for p in sorted(TEST.rglob('*')):
    if p.is_file() and p.suffix in {'.txt','.tsv'} and p.stat().st_size > 0:
        df = pd.read_csv(p, sep='\t', dtype=str)
        print(p.name, df.shape); print('columns:', list(df.columns)); break


## Cell 7 - Full run

In [ ]:
import subprocess, time
from pathlib import Path

# Results live on Drive so a reclaimed runtime does not cost the whole run.
OUT = PERSIST / 'kleborate'; OUT.mkdir(parents=True, exist_ok=True)
CHUNK = 20

def build_cmd(chunk, tag):
    f = list(map(str, chunk))
    if WORKING_CMD == 'v3 preset kpsc':
        return ['kleborate','-a',*f,'-o',str(tag),'-p','kpsc']
    if WORKING_CMD == 'v3 preset kp':
        return ['kleborate','-a',*f,'-o',str(tag),'-p','kp']
    if WORKING_CMD == 'v3 module kpsc':
        return ['kleborate','-a',*f,'-o',str(tag),'-m','klebsiella_pneumo_complex']
    return ['kleborate','-a',*f,'--all','-o',str(tag)+'.txt']

# Process in the order of the (shuffled) accession list rather than
# sorted filename order. Accession numbers cluster by submitting centre
# and region, so a sorted run that stops early leaves a geographically
# skewed subset. In list order, any prefix is still a balanced random
# sample across groups - which matters because long runs get interrupted.
fastas = [ASM / f'{a}.fna' for a in accessions if (ASM / f'{a}.fna').exists()]
chunks = [fastas[i:i+CHUNK] for i in range(0, len(fastas), CHUNK)]
print(len(fastas), 'genomes in', len(chunks), 'chunks')

failed = []; t0 = time.time()
for n, chunk in enumerate(chunks, 1):
    tag = OUT / f'chunk_{n:03d}'; done = OUT / f'chunk_{n:03d}.done'
    if done.exists():
        print(f'chunk {n}/{len(chunks)} cached'); continue
    r = subprocess.run(build_cmd(chunk, tag), capture_output=True, text=True)
    if r.returncode != 0:
        failed.append(n)
        print(f'chunk {n} FAILED:', (r.stderr or '').strip()[-500:]); continue
    done.touch(); el = time.time() - t0
    print(f'chunk {n}/{len(chunks)} ok [{el/60:.1f} min, '
          f'~{(el/n)*(len(chunks)-n)/60:.1f} min left]')

print(f'\nTotal {(time.time()-t0)/60:.1f} min; failed: {failed}')


## Cell 8 - Collect, summarise, save

In [ ]:
import pandas as pd

# Kleborate v3 writes TWO tables per run:
#   klebsiella_pneumo_complex_output.txt               <- main, 1 row/genome
#   klebsiella_pneumo_complex_hAMRonization_output.txt <- AMR long format
# Concatenating both would interleave two different schemas, so take the
# main table only. The AMR table is collected separately below.
MAIN = 'klebsiella_pneumo_complex_output.txt'
AMRH = 'hAMRonization'

frames, amr_frames = [], []
for p in sorted(OUT.rglob('*.txt')):
    if not p.is_file() or p.stat().st_size == 0:
        continue
    try:
        df = pd.read_csv(p, sep='\t', dtype=str)
    except Exception as e:
        print('skip', p.name, e); continue
    if p.name == MAIN:
        frames.append(df)
    elif AMRH in p.name:
        amr_frames.append(df)

if not frames:
    raise SystemExit(f'No {MAIN} found - check the Cell 7 output.')

kleb = pd.concat(frames, ignore_index=True).drop_duplicates()
print('main table :', kleb.shape)
print('columns:', list(kleb.columns))

if amr_frames:
    amr = pd.concat(amr_frames, ignore_index=True).drop_duplicates()
    amr.to_csv('/content/kleborate_comparison_kp_amr.csv', index=False)
    amr.to_csv(PERSIST / 'kleborate_comparison_kp_amr.csv', index=False)
    print('AMR table  :', amr.shape, '-> kleborate_comparison_kp_amr.csv')

def pick(df, *cands):
    low = {c.lower().replace(' ','_'): c for c in df.columns}
    for c in cands:
        if c in df.columns: return c
        k = c.lower().replace(' ','_')
        if k in low: return low[k]
    return None

col_strain = pick(kleb,'strain','Genome Name','Name','assembly')
col_k      = pick(kleb,'K_locus','K locus','Best match locus','K_type')
col_kconf  = pick(kleb,'K_locus_confidence','K locus confidence','Match confidence')
col_o      = pick(kleb,'O_locus','O locus','O_type')
col_st     = pick(kleb,'ST','MLST ST','st')
print('resolved:', col_strain, col_k, col_kconf, col_o, col_st)

if col_k:
    kd = kleb[col_k].fillna('unknown').value_counts()
    print(f'\n=== K-locus distribution ({kd.size} types) ===')
    print(kd.head(30).to_string())
    print(f'\nTop 10 cover {kd.head(10).sum()/kd.sum()*100:.1f}%')
for lab, c in [('K confidence',col_kconf),('O-locus',col_o),('ST',col_st)]:
    if c:
        print(f'\n=== {lab} ===')
        print(kleb[c].fillna('unknown').value_counts().head(15).to_string())

if col_strain:
    kleb['assembly'] = (kleb[col_strain].astype(str)
                        .str.replace(r'\.(fna|fa|fasta)$','',regex=True).str.strip())

# Write to Drive as well as local scratch, so the result survives even if
# the browser download in the next cell is missed.
kleb.to_csv('/content/kleborate_comparison_kp.csv', index=False)
kleb.to_csv(PERSIST / 'kleborate_comparison_kp.csv', index=False)
print('\nwrote kleborate_comparison_kp.csv', kleb.shape)
print('  -> /content/ and', PERSIST)


## Cell 9 - Download the result

In [ ]:
from google.colab import files
files.download('/content/kleborate_comparison_kp.csv')
